In [331]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [332]:
lgd = pd.read_excel(
    "../data/raw/LGD_Master_District Code.xlsx",
    header=1
)

shrug = pd.read_csv(
    "../data/interim/shrug_features.csv"
)

poverty = pd.read_csv(
    "../data/interim/poverty_features.csv"
)

night = pd.read_csv(
    "../data/interim/nightlight_features.csv"
)

mgnregs = pd.read_csv(
    "../data/interim/mgnregs_features.csv"
)

nfhs4 = pd.read_csv(
    "../data/interim/nfhs4_clean.csv"
)

nfhs5 = pd.read_csv(
    "../data/interim/nfhs5_clean.csv"
)

/Users/thalakolakarthikreddy/miniconda3/envs/dwei/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [333]:
lgd_ref = (
    lgd.rename(
        columns={
            "State Name (In English)": "State",
            "District Name(In English)": "District",
            "District Code": "lgd_district_code",
            "Census 2011 Code": "pc11_district_id"
        }
    )[
        [
            "State",
            "District",
            "lgd_district_code",
            "pc11_district_id"
        ]
    ]
)

In [334]:
lgd_ref = lgd_ref[
    lgd_ref["pc11_district_id"] != 0
].copy()

print(lgd_ref.shape)

(638, 4)


In [335]:
master = lgd_ref.merge(
    shrug,
    on="pc11_district_id",
    how="left"
)

print(master.shape)

(638, 10)


In [336]:
master = master.merge(
    poverty[
        [
            "pc11_district_id",
            "poverty_rate",
            "poverty_log"
        ]
    ],
    on="pc11_district_id",
    how="left"
)

In [337]:
master = master.merge(
    night[
        [
            "pc11_district_id",
            "night_lights_log"
        ]
    ],
    on="pc11_district_id",
    how="left"
)

In [338]:
master["state_key"] = (
    master["State"]
    .str.upper()
    .str.strip()
)

master["district_key"] = (
    master["District"]
    .str.upper()
    .str.strip()
)

In [339]:
mgnregs["state_key"] = (
    mgnregs["state_name"]
    .str.upper()
    .str.strip()
)

mgnregs["district_key"] = (
    mgnregs["district_name"]
    .str.upper()
    .str.strip()
)

DON't Touch it ⬇️

In [222]:
mapping = pd.DataFrame(
    columns=[
        "State",
        "lgd_district",
        "mgnregs_district",
        "status"
    ]
)

mapping.to_csv(
    "../data/validation/mgnregs_manual_mapping.csv",
    index=False
)

print("Created")

Created


In [340]:
mapping = pd.read_csv(
    "../data/validation/mgnregs_manual_mapping.csv"
)

mapping.head()

,State,lgd_district,mgnregs_district,status


In [341]:
print(master.shape)
print(mgnregs.shape)

(638, 15)
(716, 9)


In [342]:
mgnregs["state_key"] = mgnregs["state_key"].replace({
    "ANDAMAN AND NICOBAR":
        "ANDAMAN AND NICOBAR ISLANDS",

    "DN HAVELI AND DD":
        "THE DADRA AND NAGAR HAVELI AND DAMAN AND DIU"
})

In [343]:
mgnregs_test = master.merge(
    mgnregs[
        [
            "state_key",
            "district_key",
            "wage_timeliness_pct",
            "avg_days_per_hh",
            "women_pct",
            "persondays_per_hh"
        ]
    ],
    on=[
        "state_key",
        "district_key"
    ],
    how="left"
)

In [344]:
print(mgnregs_test.shape)

print(
    mgnregs_test[
        [
            "wage_timeliness_pct",
            "avg_days_per_hh",
            "women_pct",
            "persondays_per_hh"
        ]
    ].isna().sum()
)

(638, 19)
wage_timeliness_pct    90
avg_days_per_hh        90
women_pct              90
persondays_per_hh      90
dtype: int64


In [345]:
mgnregs_unmatched = (
    mgnregs_test[
        mgnregs_test["wage_timeliness_pct"].isna()
    ][
        ["State", "District"]
    ]
    .drop_duplicates()
    .sort_values(
        ["State", "District"]
    )
)

print(mgnregs_unmatched.shape)

(90, 2)


In [229]:
mgnregs_unmatched.to_csv(
    "../data/validation/mgnregs_unmatched.csv",
    index=False
)

In [376]:
mgnregs.loc[
    mgnregs["state_key"] == "PUDUCHERRY",
    ["district_name"]
].drop_duplicates().sort_values("district_name")

,district_name
467,KARAIKAL
466,PONDICHERRY


In [377]:
mapping = pd.read_csv(
    "../data/validation/mgnregs_manual_mapping.csv"
)

In [379]:
mapping_valid = mapping[
    mapping["status"] == "DIRECT_MATCH"
].copy()

In [380]:
mapping_valid["lgd_district"] = (
    mapping_valid["lgd_district"]
    .str.upper()
    .str.strip()
)

mapping_valid["mgnregs_district"] = (
    mapping_valid["mgnregs_district"]
    .str.upper()
    .str.strip()
)

In [381]:
mapping_dict = dict(
    zip(
        mapping_valid["lgd_district"],
        mapping_valid["mgnregs_district"]
    )
)

In [382]:
master["district_merge"] = (
    master["district_key"]
    .replace(mapping_dict)
)

mgnregs["district_merge"] = (
    mgnregs["district_key"]
)

In [383]:
master_mgnregs = master.merge(
    mgnregs[
        [
            "state_key",
            "district_merge",
            "wage_timeliness_pct",
            "avg_days_per_hh",
            "women_pct",
            "persondays_per_hh"
        ]
    ],
    on=[
        "state_key",
        "district_merge"
    ],
    how="left"
)

In [384]:
master_mgnregs[
    [
        "wage_timeliness_pct",
        "avg_days_per_hh",
        "women_pct",
        "persondays_per_hh"
    ]
].isna().sum()

wage_timeliness_pct    18
avg_days_per_hh        18
women_pct              18
persondays_per_hh      18
dtype: int64

In [385]:
remaining = (
    master_mgnregs[
        master_mgnregs["wage_timeliness_pct"].isna()
    ][
        ["State", "District"]
    ]
    .drop_duplicates()
    .sort_values(
        ["State", "District"]
    )
)

print(remaining.shape)

remaining

(18, 2)


,State,District
97,Chandigarh,Chandigarh
116,Delhi,Central
117,Delhi,East
118,Delhi,New Delhi
119,Delhi,North
120,Delhi,North East
121,Delhi,North West
122,Delhi,South
123,Delhi,South West
124,Delhi,West


In [386]:
master_mgnregs.to_csv(
    "../data/master/master_mgnregs.csv",
    index=False
)
master_mgnregs.to_parquet(
    "../data/master/master_mgnregs.parquet",
    index=False
)

MGNREGS Harmonization Summary

Total LGD districts: 638

Successfully matched: 621

Unmatched: 17

Reasons:
- Delhi district geographies absent from MGNREGS
- Chandigarh absent from MGNREGS
- Major metropolitan districts (Mumbai, Chennai, Hyderabad, Kolkata)
- UT restructuring cases (Daman, Diu)

Final coverage: 97.34%

NFHS4 integration

In [387]:
print(nfhs4.shape)
nfhs4.columns.tolist()

(636, 10)


['State',
 'District',
 'STUNTING',
 'WASTING',
 'UNDERWEIGHT',
 'ANEMIA',
 'INST_DEL',
 'SANITATION',
 'CLEAN_FUEL',
 'IMMUNIZATION']

In [388]:
print(nfhs5.shape)
nfhs5.columns.tolist()

(704, 10)


['State',
 'District',
 'STUNTING',
 'WASTING',
 'UNDERWEIGHT',
 'ANEMIA',
 'INST_DEL',
 'SANITATION',
 'CLEAN_FUEL',
 'IMMUNIZATION']

In [389]:
nfhs4["state_key"] = (
    nfhs4["State"]
    .str.upper()
    .str.strip()
)

nfhs4["district_key"] = (
    nfhs4["District"]
    .str.upper()
    .str.strip()
)
nfhs5["state_key"] = (
    nfhs5["State"]
    .str.upper()
    .str.strip()
)

nfhs5["district_key"] = (
    nfhs5["District"]
    .str.upper()
    .str.strip()
)

In [390]:
master_mgnregs["state_key"] = (
    master_mgnregs["State"]
    .str.upper()
    .str.strip()
)

master_mgnregs["district_key"] = (
    master_mgnregs["District"]
    .str.upper()
    .str.strip()
)

NFHS UNMATCHED

In [393]:
nfhs4.loc[
    nfhs4["State"] == "Puducherry",
    ["District"]
].sort_values("District")

,District
424,Karaikal
425,Mahe
426,Pondicherry
427,Yanam


MERGING NFHS DATA

In [394]:
nfhs_mapping = pd.read_csv(
    "../data/validation/nfhs_manual_mapping.csv"
)

nfhs_mapping.head()

,State,master_district,nfhs_district,status
0,Andhra Pradesh,Y.S.R. Kadapa,Y.S.R.,DIRECT_MATCH
1,Assam,Sribhumi,Karimganj,DIRECT_MATCH
2,Chandigarh,Chandigarh,NaN,NO_NFHS_DATA
3,Gujarat,Ahmedabad,Ahmadabad,DIRECT_MATCH
4,Gujarat,Dahod,Dohad,DIRECT_MATCH


In [395]:
nfhs_mapping_valid = (
    nfhs_mapping[
        nfhs_mapping["status"] == "DIRECT_MATCH"
    ]
    .copy()
)

In [396]:
nfhs_mapping_dict = dict(
    zip(
        nfhs_mapping_valid["master_district"]
        .str.upper()
        .str.strip(),

        nfhs_mapping_valid["nfhs_district"]
        .str.upper()
        .str.strip()
    )
)

In [397]:
master_mgnregs["district_nfhs"] = (
    master_mgnregs["district_key"]
    .replace(nfhs_mapping_dict)
)

nfhs4["district_nfhs"] = (
    nfhs4["district_key"]
)


nfhs5["district_nfhs"] = (
    nfhs5["district_key"]
)


In [398]:
nfhs4_features = (
    nfhs4.rename(
        columns={
            "STUNTING": "STUNTING_NFHS4",
            "WASTING": "WASTING_NFHS4",
            "UNDERWEIGHT": "UNDERWEIGHT_NFHS4",
            "ANEMIA": "ANEMIA_NFHS4",
            "INST_DEL": "INST_DEL_NFHS4",
            "SANITATION": "SANITATION_NFHS4",
            "CLEAN_FUEL": "CLEAN_FUEL_NFHS4",
            "IMMUNIZATION": "IMMUNIZATION_NFHS4"
        }
    )
)

nfhs5_features = (
    nfhs5.rename(
        columns={
            "STUNTING": "STUNTING_NFHS5",
            "WASTING": "WASTING_NFHS5",
            "UNDERWEIGHT": "UNDERWEIGHT_NFHS5",
            "ANEMIA": "ANEMIA_NFHS5",
            "INST_DEL": "INST_DEL_NFHS5",
            "SANITATION": "SANITATION_NFHS5",
            "CLEAN_FUEL": "CLEAN_FUEL_NFHS5",
            "IMMUNIZATION": "IMMUNIZATION_NFHS5"
        }
    )
)

In [399]:
master_nfhs4 = master_mgnregs.merge(
    nfhs4_features[
        [
            "state_key",
            "district_nfhs",
            "STUNTING_NFHS4",
            "WASTING_NFHS4",
            "UNDERWEIGHT_NFHS4",
            "ANEMIA_NFHS4",
            "INST_DEL_NFHS4",
            "SANITATION_NFHS4",
            "CLEAN_FUEL_NFHS4",
            "IMMUNIZATION_NFHS4"
        ]
    ],
    on=[
        "state_key",
        "district_nfhs"
    ],
    how="left"
)

In [400]:
master_nfhs4[
    [
        "STUNTING_NFHS4"
    ]
].isna().sum()

STUNTING_NFHS4    5
dtype: int64

In [401]:
master_final = master_nfhs4.merge(
    nfhs5_features[
        [
            "state_key",
            "district_nfhs",
            "STUNTING_NFHS5",
            "WASTING_NFHS5",
            "UNDERWEIGHT_NFHS5",
            "ANEMIA_NFHS5",
            "INST_DEL_NFHS5",
            "SANITATION_NFHS5",
            "CLEAN_FUEL_NFHS5",
            "IMMUNIZATION_NFHS5"
        ]
    ],
    on=[
        "state_key",
        "district_nfhs"
    ],
    how="left"
)

In [413]:
master_final.to_parquet(
    "../data/master/master_dataset.parquet",
    index=False
)

master_final.to_csv(
    "../data/master/master_dataset.csv",
    index=False
)

In [414]:
missing_report = master_final.isna().sum()

missing_report.to_csv(
    "../data/validation/missingness_report.csv"
)

LGD Bridge                      ✓
SHRUG Merge                     ✓
Poverty Merge                   ✓
Night Lights Merge              ✓
MGNREGS Harmonization           ✓
NFHS4 Harmonization             ✓
NFHS5 Harmonization             ✓
Missing Value Audit             ✓
Master Dataset Creation         ✓